# Notebook 08: Bronze snapshot retention

## Runtime contract

- **Precondition:** run only after end-to-end validation succeeds.
- **Parameter:** required pipeline `processing_date` in `yyyy-MM-dd`.
- **Retention behavior:** retain the current run-date snapshot for replay, retain future-dated snapshots, and delete only older dated files matching the expected table naming pattern.
- **Output:** append-only per-file evidence in `audit.bronze_cleanup`.
- **Failure behavior:** audit evidence is persisted before the notebook fails for missing current snapshots, delete verification errors, or an empty cleanup scope.

In [ ]:
processing_date = ""


In [ ]:
from datetime import datetime, timezone
from pathlib import PurePosixPath
import re
import uuid

from pyspark.sql import types as T

BRZ_SLV_LH = "lh_brz_slv_finance"
BRONZE_ROOT = "Files/bronze/sap_finance"
AUDIT_TABLE = f"{BRZ_SLV_LH}.audit.bronze_cleanup"
TABLES = [
    "acdoca", "bkpf", "t001", "csks", "cepc", "ska1", "skat",
    "lfa1", "kna1", "faglflexp", "budget", "division_reference",
    "gl_account_reference",
]

processing_date = processing_date.strip()
if not processing_date:
    raise ValueError("processing_date is required when cleanup runs from the pipeline")
try:
    current_date = datetime.strptime(processing_date, "%Y-%m-%d").date()
except ValueError as exc:
    raise ValueError("processing_date must use yyyy-MM-dd format") from exc

cleanup_run_id = str(uuid.uuid4())
context = notebookutils.runtime.context
pipeline_activity_id = str(context.get("activityId", ""))
completed_utc = datetime.now(timezone.utc).replace(tzinfo=None)
file_pattern = re.compile(r"^(?P<table>[a-z0-9_]+)_(?P<date>\d{8})\.csv$", re.IGNORECASE)
rows = []
errors = []

for table_name in TABLES:
    folder = f"{BRONZE_ROOT}/{table_name}"
    for file_info in notebookutils.fs.ls(folder):
        file_name = PurePosixPath(file_info.path).name
        match = file_pattern.fullmatch(file_name)
        if not match or match.group("table").lower() != table_name:
            continue
        file_date = datetime.strptime(match.group("date"), "%Y%m%d").date()
        size_bytes = int(getattr(file_info, "size", 0) or 0)
        if file_date == current_date:
            rows.append(
                (
                    cleanup_run_id, current_date, table_name, file_info.path,
                    file_name, file_date, size_bytes, "RETAIN",
                    "RETAINED_CURRENT", "", completed_utc, pipeline_activity_id,
                )
            )
            continue
        if file_date > current_date:
            rows.append(
                (
                    cleanup_run_id, current_date, table_name, file_info.path,
                    file_name, file_date, size_bytes, "RETAIN",
                    "RETAINED_FUTURE", "", completed_utc, pipeline_activity_id,
                )
            )
            continue
        try:
            notebookutils.fs.rm(file_info.path, False)
            if notebookutils.fs.exists(file_info.path):
                raise RuntimeError("File still exists after delete")
            rows.append(
                (
                    cleanup_run_id, current_date, table_name, file_info.path,
                    file_name, file_date, size_bytes, "DELETE", "DELETED",
                    "", completed_utc, pipeline_activity_id,
                )
            )
        except Exception as exc:
            message = str(exc)[:1000]
            rows.append(
                (
                    cleanup_run_id, current_date, table_name, file_info.path,
                    file_name, file_date, size_bytes, "DELETE", "ERROR",
                    message, completed_utc, pipeline_activity_id,
                )
            )
            errors.append(f"{file_info.path}: {message}")

schema = T.StructType([
    T.StructField("CLEANUP_RUN_ID", T.StringType(), False),
    T.StructField("PROCESSING_DATE", T.DateType(), False),
    T.StructField("TABLE_NAME", T.StringType(), False),
    T.StructField("SOURCE_PATH", T.StringType(), False),
    T.StructField("SOURCE_FILE_NAME", T.StringType(), False),
    T.StructField("FILE_DATE", T.DateType(), False),
    T.StructField("FILE_SIZE_BYTES", T.LongType(), False),
    T.StructField("ACTION", T.StringType(), False),
    T.StructField("STATUS", T.StringType(), False),
    T.StructField("ERROR_MESSAGE", T.StringType(), False),
    T.StructField("COMPLETED_UTC", T.TimestampType(), False),
    T.StructField("PIPELINE_ACTIVITY_ID", T.StringType(), False),
])

if not rows:
    raise RuntimeError(f"No dated Bronze snapshots found for processing_date={processing_date}")

audit_df = spark.createDataFrame(rows, schema)
audit_df.write.format("delta").mode("append").saveAsTable(AUDIT_TABLE)
display(audit_df.orderBy("TABLE_NAME", "FILE_DATE"))

current_files = audit_df.filter("STATUS = 'RETAINED_CURRENT'").count()
if current_files != len(TABLES):
    raise RuntimeError(f"Expected {len(TABLES)} current snapshots, found {current_files}")
if errors:
    raise RuntimeError("Bronze cleanup failed: " + " | ".join(errors))
